# Fine-Tuning, and What It Destroys

The full code of [Fine-Tuning, and What It Destroys](https://residual-stream.lvtan71.workers.dev/posts/fine-tuning-and-forgetting/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/fine-tuning-and-forgetting) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/fine-tuning-and-forgetting.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 24 min end to end on the RTX 3090 machine that saved the outputs below.
Fine-tunes Qwen2.5-0.5B eight times (four settings, two seeds, 400 steps each). Downloads two Wikipedia shards (about 0.7 GB). The table is made by a short script the post doesn't show, included here. At lr 1e-4 training is unstable, so those rows vary by machine, by up to 0.11 bits per byte.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 23 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "transformers==5.18.*" huggingface_hub pandas pyarrow

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "fine-tuning-and-forgetting"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/fine-tuning-and-forgetting


## `sft_loss.py`

One chat example through Qwen2.5's chat template, with the prompt's tokens masked to -100 so only the answer is trained.

In [2]:
!{sys.executable} sft_loss.py

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What does the KV cache store?<|im_end|>
<|im_start|>assistant
The keys and values of every earlier token, so they aren't recomputed.<|im_end|>

54 tokens: 36 of prompt (system, user, assistant header), masked; 18 trained:
"The keys and values of every earlier token, so they aren't recomputed.<|im_end|>\n"
targets in the loss: 18 of 53


The post printed:

```text
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What does the KV cache store?<|im_end|>
<|im_start|>assistant
The keys and values of every earlier token, so they aren't recomputed.<|im_end|>

54 tokens: 36 of prompt (system, user, assistant header), masked; 18 trained:
"The keys and values of every earlier token, so they aren't recomputed.<|im_end|>\n"
targets in the loss: 18 of 53
```

## `fetch_wiki.py`

Downloads one shard each of English and Vietnamese Wikipedia (the November 2023 dump) into `wiki/`.

In [3]:
!{sys.executable} fetch_wiki.py

## `wiki_data.py`

Builds the replay text (`wiki_replay.txt`, about 4 million characters of English Wikipedia, minus any article whose title is in WikiText-2's test set) and the Vietnamese test text (`vi_test.txt`, 120,000 characters). Reads `wiki/`.

In [4]:
!{sys.executable} wiki_data.py

WikiText test articles: 7, of which 0 are in this shard and dropped; replay text 4,001,877 chars from 1101 articles


The post printed:

```text
WikiText test articles: 7, of which 0 are in this shard and dropped; replay text 4,001,877 chars from 1101 articles
```

## `finetune_all.py`

Fine-tunes Qwen2.5-0.5B eight times with `ft.py` (learning rates 1e-5 and 1e-4, with and without 25% replay, two seeds, 400 steps each) and keeps each run's summary line in `ft.out`. It printed those eight lines, which were not kept, so it has no `.out`.

In [5]:
!{sys.executable} finetune_all.py

lr 1e-05, replay 0%, seed 0 (161 s): step 0: blog 1.163, wiki 0.981, vi 1.022 | step 25: blog 1.112, wiki 0.990, vi 1.028 | step 50: blog 1.087, wiki 1.002, vi 1.035 | step 100: blog 1.072, wiki 1.013, vi 1.042 | step 200: blog 1.071, wiki 1.031, vi 1.055 | step 400: blog 1.068, wiki 1.048, vi 1.073


lr 0.0001, replay 0%, seed 0 (152 s): step 0: blog 1.163, wiki 0.981, vi 1.022 | step 25: blog 1.196, wiki 1.122, vi 1.113 | step 50: blog 1.217, wiki 1.165, vi 1.147 | step 100: blog 1.246, wiki 1.260, vi 1.248 | step 200: blog 1.262, wiki 1.355, vi 1.447 | step 400: blog 1.309, wiki 1.474, vi 1.513


lr 0.0001, replay 25%, seed 0 (164 s): step 0: blog 1.163, wiki 0.981, vi 1.022 | step 25: blog 1.193, wiki 1.082, vi 1.088 | step 50: blog 1.227, wiki 1.167, vi 1.146 | step 100: blog 1.252, wiki 1.238, vi 1.244 | step 200: blog 1.261, wiki 1.295, vi 1.434 | step 400: blog 1.308, wiki 1.332, vi 1.463


lr 1e-05, replay 25%, seed 0 (160 s): step 0: blog 1.163, wiki 0.981, vi 1.022 | step 25: blog 1.118, wiki 0.985, vi 1.022 | step 50: blog 1.094, wiki 0.988, vi 1.021 | step 100: blog 1.078, wiki 0.988, vi 1.021 | step 200: blog 1.073, wiki 0.996, vi 1.024 | step 400: blog 1.061, wiki 0.997, vi 1.027


lr 1e-05, replay 0%, seed 1 (159 s): step 0: blog 1.163, wiki 0.981, vi 1.022 | step 25: blog 1.113, wiki 0.990, vi 1.030 | step 50: blog 1.092, wiki 1.004, vi 1.036 | step 100: blog 1.073, wiki 1.012, vi 1.042 | step 200: blog 1.065, wiki 1.027, vi 1.052 | step 400: blog 1.085, wiki 1.052, vi 1.073


lr 0.0001, replay 0%, seed 1 (157 s): step 0: blog 1.163, wiki 0.981, vi 1.022 | step 25: blog 1.215, wiki 1.103, vi 1.105 | step 50: blog 1.224, wiki 1.159, vi 1.135 | step 100: blog 1.226, wiki 1.223, vi 1.220 | step 200: blog 1.273, wiki 1.348, vi 1.611 | step 400: blog 1.329, wiki 1.460, vi 1.549


lr 0.0001, replay 25%, seed 1 (156 s): step 0: blog 1.163, wiki 0.981, vi 1.022 | step 25: blog 1.213, wiki 1.113, vi 1.088 | step 50: blog 1.233, wiki 1.170, vi 1.177 | step 100: blog 1.233, wiki 1.224, vi 1.224 | step 200: blog 1.251, wiki 1.276, vi 1.297 | step 400: blog 1.278, wiki 1.335, vi 1.359


lr 1e-05, replay 25%, seed 1 (155 s): step 0: blog 1.163, wiki 0.981, vi 1.022 | step 25: blog 1.117, wiki 0.982, vi 1.022 | step 50: blog 1.097, wiki 0.989, vi 1.020 | step 100: blog 1.079, wiki 0.990, vi 1.021 | step 200: blog 1.066, wiki 0.993, vi 1.022 | step 400: blog 1.069, wiki 0.997, vi 1.026


## `ft_table.py`

Turns `ft.out` into the post's table: bits per byte on the held-out blog posts, WikiText-2 and Vietnamese Wikipedia at each checkpoint.

In [6]:
!{sys.executable} ft_table.py

before training: blog 1.163, WikiText 0.981, Vietnamese 1.022 bits per byte

bits per byte after each step, seed 0 then seed 1
lr      replay         step 25       step 50       step 100      step 200      step 400
1e-05     0%  blog     1.112 1.113   1.087 1.092   1.072 1.073   1.071 1.065   1.068 1.085
1e-05     0%  wiki     0.990 0.990   1.002 1.004   1.013 1.012   1.031 1.027   1.048 1.052
1e-05     0%  vi       1.028 1.030   1.035 1.036   1.042 1.042   1.055 1.052   1.073 1.073
1e-05    25%  blog     1.118 1.117   1.094 1.097   1.078 1.079   1.073 1.066   1.061 1.069
1e-05    25%  wiki     0.985 0.982   0.988 0.989   0.988 0.990   0.996 0.993   0.997 0.997
1e-05    25%  vi       1.022 1.022   1.021 1.020   1.021 1.021   1.024 1.022   1.027 1.026
0.0001    0%  blog     1.196 1.215   1.217 1.224   1.246 1.226   1.262 1.273   1.309 1.329
0.0001    0%  wiki     1.122 1.103   1.165 1.159   1.260 1.223   1.355 1.348   1.474 1.460
0.0001    0%  vi       1.113 1.105   1.147 1.135   1.248 

The post printed:

```text
before training: blog 1.163, WikiText 0.981, Vietnamese 1.022 bits per byte

bits per byte after each step, seed 0 then seed 1
lr      replay         step 25       step 50       step 100      step 200      step 400
1e-05     0%  blog     1.112 1.113   1.086 1.092   1.072 1.072   1.069 1.065   1.068 1.084
1e-05     0%  wiki     0.990 0.990   1.002 1.004   1.012 1.011   1.031 1.026   1.047 1.050
1e-05     0%  vi       1.028 1.029   1.035 1.035   1.042 1.042   1.054 1.052   1.071 1.072
1e-05    25%  blog     1.118 1.117   1.094 1.097   1.077 1.079   1.072 1.066   1.060 1.068
1e-05    25%  wiki     0.985 0.982   0.988 0.988   0.989 0.989   0.996 0.993   0.997 0.997
1e-05    25%  vi       1.022 1.022   1.021 1.020   1.021 1.021   1.024 1.022   1.026 1.025
0.0001    0%  blog     1.197 1.214   1.219 1.217   1.247 1.241   1.256 1.259   1.311 1.357
0.0001    0%  wiki     1.124 1.101   1.159 1.151   1.253 1.232   1.371 1.338   1.460 1.513
0.0001    0%  vi       1.114 1.105   1.149 1.136   1.251 1.223   1.400 1.506   1.502 1.609
0.0001   25%  blog     1.193 1.210   1.228 1.230   1.254 1.228   1.263 1.238   1.294 1.292
0.0001   25%  wiki     1.082 1.111   1.165 1.165   1.239 1.219   1.291 1.284   1.320 1.351
0.0001   25%  vi       1.088 1.088   1.146 1.178   1.239 1.226   1.572 1.293   1.455 1.400
```